# kaiming-uniform-init — worked example 2: Kaiming bound shrinks as fan_in grows — empirical comparison across widths

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-init`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import math
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The Kaiming-uniform bound `1/sqrt(fan_in)` decreases as the number of input features grows. A wider preceding layer produces a smaller initial weight magnitude. This keeps the variance of the layer's output roughly constant regardless of layer width — the central motivation behind the Kaiming initialization scheme.

## Worked solution

Step 1: For each of three `in_features` values (4, 64, 1024), compute the theoretical bound `1/sqrt(in_features)` and build a weight tensor by calling `nn.init.kaiming_uniform_` with `mode='fan_in', a=math.sqrt(5)` — the exact call PyTorch's `nn.Linear.reset_parameters` uses.

Step 2: Check empirically that `weight.abs().max() <= bound` for each case (uniform distributions are bounded).

Step 3: Print all three (bound, empirical_max) pairs to show the shrinking trend. Also print the ratio `empirical_max / bound` — it should be close to 1.0, confirming the samples explore the full range.

In [ ]:
import torch as t
import torch.nn as nn
import math

t.manual_seed(42)

def kaiming_init_at_width(in_features: int, out_features: int = 128):
    """Return (bound, weight_tensor) for fan_in = in_features."""
    bound = 1.0 / math.sqrt(in_features)
    w = t.empty(out_features, in_features)
    # This is exactly what nn.Linear.reset_parameters calls:
    nn.init.kaiming_uniform_(w, a=math.sqrt(5))
    return bound, w

widths = [4, 64, 1024]
print(f'{'fan_in':>8}  {'bound':>8}  {'emp_max':>8}  {'ratio':>7}')
for fan_in in widths:
    bound, w = kaiming_init_at_width(fan_in)
    emp_max = w.abs().max().item()
    ratio = emp_max / bound
    within = emp_max <= bound + 1e-6
    print(f'{fan_in:>8}  {bound:>8.4f}  {emp_max:>8.4f}  {ratio:>7.3f}  within={within}')

print()
print('Bound halves roughly every 4x increase in fan_in (1/sqrt relationship).')